In [1]:
import logging
import os.path
import time
from concurrent.futures.process import ProcessPoolExecutor
from concurrent.futures.thread import ThreadPoolExecutor
from typing import Dict
import sys
import pandas as pd

sys.path.append("../../")
import biked_commons

from biked_commons.api.rendering import RenderingEngine, FILE_BUILDER
from biked_commons.resource_utils import resource_path, STANDARD_BIKE_RESOURCE, split_datasets_path

# Configure the logging
logging.basicConfig(level=logging.DEBUG,  # Set the logging level to DEBUG
                    format='%(asctime)s - %(levelname)s - %(message)s')  # Customize the log format

Using java as the Java binary


In [6]:
def read_standard_xml():
    with open(STANDARD_BIKE_RESOURCE, "r") as file:
        return file.read()

standard_bike_xml = read_standard_xml()

def get_records_with_id() -> Dict[str, dict]:
    """
    Return records in a dictionary of the form {
    (record_id: str) : (record: dict)
    }
    """
    data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0).iloc[:100]

    return {str(record_id): record for record_id, record in zip(data.index.tolist(), data.to_dict(orient="records"))}

def record_to_xml(save_path: str, record_id: str, record: dict):
    try:
        file_path = os.path.join(save_path, f"{record_id}.xml")
        print(f"Writing {file_path}")
        with open(file_path, "w") as file:
            xml_data = FILE_BUILDER.build_cad_from_clip(record, standard_bike_xml, False)
            file.write(xml_data)
    except Exception as e:
        print(f"Failed with exception {e}")


def convert_to_xml(records_with_id: Dict[str, dict],
                   process_pool_workers: int,
                   save_dir: str
                   ):
    executor = ThreadPoolExecutor(max_workers=process_pool_workers)
    os.makedirs(save_dir, exist_ok=True)
    for record_id, record in records_with_id.items():
        executor.submit(record_to_xml, save_dir, record_id, record)
    executor.shutdown()  # waits for all submitted tasks to finish


In [7]:
records = get_records_with_id()
process_pool_workers = 1
xml_dir = os.path.join(resource_path("bike_bench_rendering"), "xml")
if __name__ == "__main__":
    start_time = time.time()
    convert_to_xml(records, process_pool_workers, xml_dir)
    end_time = time.time()
    logging.info(f"Conversion completed in {end_time - start_time} seconds")

Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\1.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\2.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\3.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\4.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\5.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\bike_embedding\../..\biked_commons\..\resources\bike_bench_rendering\xml\6.xml
Writing c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\b

2025-05-01 21:05:09,041 - INFO - Conversion completed in 101.33704280853271 seconds


In [17]:


def run_rendering_benchmark(
        number_rendering_servers: int,
        thread_pool_workers: int,
        records_with_id: Dict[str, dict],
        xml_dir: str,
        image_dir: str,
):
    executor = ThreadPoolExecutor(max_workers=thread_pool_workers)
    rendering_engine = RenderingEngine(number_rendering_servers=number_rendering_servers, server_init_timeout_seconds=30)

    os.makedirs(image_dir, exist_ok=True)
    def render_record(xml: str):
        try:
            xml_path = os.path.join(xml_dir, f"{xml}.xml")
            with open(xml_path, "r") as xml_file:
                # print("Sending request to server...")
                read_file = xml_file.read()
                # print("Read file...")
                rendering_result = rendering_engine.render_xml(read_file)
                # print("Rendering result received from server...")
                image_path = os.path.join(image_dir, f"{xml}.png")
                with open(image_path, "wb") as image_file:
                    image_file.write(rendering_result.image_bytes)
                    print("Image file written to disk.")
                return True
        except Exception as e:
            print(f"Rendering failed: {e}")
            return False, e

    for record_id, _ in records_with_id.items():
        executor.submit(render_record, record_id)

    executor.shutdown()


In [20]:
NUMBER_SERVERS = 12

THREAD_POOL_WORKERS = 12

xml_dir = os.path.join(resource_path("bike_bench_rendering"), "xml")
image_dir = os.path.join(resource_path("bike_bench_rendering"), "images")

run_rendering_benchmark(
    number_rendering_servers=NUMBER_SERVERS,
    thread_pool_workers=THREAD_POOL_WORKERS,
    records_with_id=records,
    xml_dir=xml_dir,
    image_dir=image_dir,
)

2025-05-01 21:27:39,505 - DEBUG - Starting new HTTP connection (1): localhost:8080
2025-05-01 21:27:39,507 - DEBUG - Starting new HTTP connection (1): localhost:8081
2025-05-01 21:27:39,508 - DEBUG - Starting new HTTP connection (1): localhost:8082
2025-05-01 21:27:39,509 - DEBUG - Starting new HTTP connection (1): localhost:8084
2025-05-01 21:27:39,510 - DEBUG - Starting new HTTP connection (1): localhost:8083
2025-05-01 21:27:39,511 - DEBUG - Starting new HTTP connection (1): localhost:8085
2025-05-01 21:27:39,512 - DEBUG - Starting new HTTP connection (1): localhost:8086
2025-05-01 21:27:39,514 - DEBUG - http://localhost:8080 "GET /actuator/serverInformation HTTP/1.1" 200 None
2025-05-01 21:27:39,514 - DEBUG - Starting new HTTP connection (1): localhost:8087
2025-05-01 21:27:39,517 - DEBUG - Starting new HTTP connection (1): localhost:8088
2025-05-01 21:27:39,517 - DEBUG - http://localhost:8081 "GET /actuator/serverInformation HTTP/1.1" 200 None
2025-05-01 21:27:39,518 - DEBUG - Sta

Image file written to disk.
Image file written to disk.
Image file written to disk.
Image file written to disk.


2025-05-01 21:27:43,314 - DEBUG - http://localhost:8084 "POST /api/v1/render HTTP/1.1" 200 1097280
2025-05-01 21:27:43,319 - DEBUG - Starting new HTTP connection (1): localhost:8084
2025-05-01 21:27:43,485 - DEBUG - http://localhost:8089 "POST /api/v1/render HTTP/1.1" 200 1114461
2025-05-01 21:27:43,490 - DEBUG - Starting new HTTP connection (1): localhost:8085


Image file written to disk.
Image file written to disk.


2025-05-01 21:27:43,521 - DEBUG - http://localhost:8090 "POST /api/v1/render HTTP/1.1" 200 1011828
2025-05-01 21:27:43,525 - DEBUG - Starting new HTTP connection (1): localhost:8086
2025-05-01 21:27:43,602 - DEBUG - http://localhost:8091 "POST /api/v1/render HTTP/1.1" 200 1107765
2025-05-01 21:27:43,607 - DEBUG - Starting new HTTP connection (1): localhost:8087
2025-05-01 21:27:43,660 - DEBUG - http://localhost:8088 "POST /api/v1/render HTTP/1.1" 200 1125129
2025-05-01 21:27:43,664 - DEBUG - http://localhost:8086 "POST /api/v1/render HTTP/1.1" 200 1072943
2025-05-01 21:27:43,665 - DEBUG - Starting new HTTP connection (1): localhost:8088
2025-05-01 21:27:43,671 - DEBUG - Starting new HTTP connection (1): localhost:8089


...

Image file written to disk.
Image file written to disk.
Image file written to disk.


2025-05-01 21:28:02,800 - DEBUG - http://localhost:8082 "POST /api/v1/render HTTP/1.1" 200 1146657
2025-05-01 21:28:02,979 - DEBUG - http://localhost:8091 "POST /api/v1/render HTTP/1.1" 200 1111311


Image file written to disk.


2025-05-01 21:28:03,003 - DEBUG - http://localhost:8083 "POST /api/v1/render HTTP/1.1" 200 1013476


Image file written to disk.
Image file written to disk.


In [24]:
#read image 1 as numpy
image_path = os.path.join(image_dir, "1.png")
import numpy as np
from PIL import Image
image = Image.open(image_path)
image = image.convert("RGB")
image_array = np.array(image)





2025-05-01 21:29:45,430 - DEBUG - Importing BlpImagePlugin
2025-05-01 21:29:45,432 - DEBUG - Importing BmpImagePlugin
2025-05-01 21:29:45,433 - DEBUG - Importing BufrStubImagePlugin
2025-05-01 21:29:45,434 - DEBUG - Importing CurImagePlugin
2025-05-01 21:29:45,435 - DEBUG - Importing DcxImagePlugin
2025-05-01 21:29:45,437 - DEBUG - Importing DdsImagePlugin
2025-05-01 21:29:45,438 - DEBUG - Importing EpsImagePlugin
2025-05-01 21:29:45,471 - DEBUG - Importing FitsImagePlugin
2025-05-01 21:29:45,473 - DEBUG - Importing FitsStubImagePlugin
2025-05-01 21:29:45,475 - DEBUG - Importing FliImagePlugin
2025-05-01 21:29:45,476 - DEBUG - Importing FpxImagePlugin
2025-05-01 21:29:45,478 - DEBUG - Image: failed to import FpxImagePlugin: No module named 'olefile'
2025-05-01 21:29:45,478 - DEBUG - Importing FtexImagePlugin
2025-05-01 21:29:45,480 - DEBUG - Importing GbrImagePlugin
2025-05-01 21:29:45,481 - DEBUG - Importing GifImagePlugin
2025-05-01 21:29:45,481 - DEBUG - Importing GribStubImagePlugi

UnidentifiedImageError: cannot identify image file 'c:\\Users\\Lyle\\Documents\\Files\\DeCoDE\\biked-commons\\src\\biked_commons\\bike_embedding\\../..\\biked_commons\\..\\resources\\bike_bench_rendering\\images\\1.png'

In [23]:
image_data

b'<?xml version="1.0" encoding="UTF-8"?>\r\n<!DOCTYPE svg PUBLIC \'-//W3C//DTD SVG 1.0//EN\'\r\n          \'http://www.w3.org/TR/2001/REC-SVG-20010904/DTD/svg10.dtd\'>\r\n<svg xmlns:xlink="http://www.w3.org/1999/xlink" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:\'Dialog\'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="1064" height="670" xmlns="http://www.w3.org/2000/svg"\r\n><!--Generated by the Batik Graphics2D SVG Generator--><defs id="genericDefs"\r\n  /><g\r\n  ><defs id="defs1"\r\n    ><clipPath clipPathUnits="userSpaceOnUse" id="clipPath1"\r\n      ><path d="M0 0 L1064 0 L1064 670 L0 670 L0 0 Z"\r\n      /></clipPath\r\n      ><clipPath clipPathUnits="userSpaceOnUse" id="clipPath2"\r\n      